In [3]:
# 1. Install library ekosistem LangChain dan pemroses PDF
!pip install langchain langchain-community langchain-core pypdf sentence-transformers chromadb pandas -U

import os
import pandas as pd
from google.colab import drive
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Sambungkan ke Google Drive kamu
drive.mount('/content/drive')

# 3. Kunci PATH folder tempat kamu menaruh file-file PDF panduan aroma tadi
# (Sesuaikan susunan nama foldernya jika ada perbedaan ketikan)
PATH_DATABASE_PDF = "/content/drive/MyDrive/Project_NLP/Project_kelompok_UAS/Data/database_rag_parfum/"

print("⚙️ Ekosistem LangChain berhasil dipasang dan Google Drive telah terhubung!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚙️ Ekosistem LangChain berhasil dipasang dan Google Drive telah terhubung!


In [4]:
print("🔄 Memulai pemindaian folder database RAG...")

# 1. Menggunakan DirectoryLoader untuk menyedot semua file .pdf secara otomatis
loader = DirectoryLoader(
    path=PATH_DATABASE_PDF,
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

# 2. AI membaca isi seluruh halaman dari semua PDF yang ada
dokumen_mentah = loader.load()
print(f"✅ Sukses! Sistem RAG berhasil membaca total {len(dokumen_mentah)} halaman dari file PDF-mu.")

# ========================================================
# 3. TAHAP TEXT CHUNKING (Memotong Kalimat Menjadi Serpihan)
# ========================================================
# Komputer tidak bisa membaca 1 buku utuh sekaligus, teks harus dipotong-potong
# menjadi fragmen kecil (chunk) berukuran 500 karakter dengan overlap 50 karakter
# agar informasi di ujung potongan tidak hilang.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

serpihan_teks = text_splitter.split_documents(dokumen_mentah)
print(f"✂️ Hasil Pemotongan: Teks PDF berhasil dipecah menjadi {len(serpihan_teks)} serpihan informasi kecil.")
print("👀 Contoh isi serpihan teks pertama:")
print(serpihan_teks[0].page_content)


🔄 Memulai pemindaian folder database RAG...
✅ Sukses! Sistem RAG berhasil membaca total 136 halaman dari file PDF-mu.
✂️ Hasil Pemotongan: Teks PDF berhasil dipecah menjadi 618 serpihan informasi kecil.
👀 Contoh isi serpihan teks pertama:
DOKUMEN STANDAR OPERASIONAL PROSEDUR (SOP): RUMUS TAKARAN 
FORMULASI 
 
Setiap formula parfum kustom berukuran total 50 ml tipe Eau de Parfum (EDP) 
wajib menggunakan komposisi: Konsentrat Minyak Esensial murni 20% (10 ml / 
200 tetes) dan Cairan Alkohol Pelarut Khusus 80% (40 ml).


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("🔄 Sedang mengunduh model embedding dan memproses 616 serpihan teks...")

# 1. Menggunakan model embedding Bahasa Indonesia/Multilingual yang sangat ringan dan akurat
# Model ini akan menerjemahkan arti kalimat menjadi koordinat angka vektor matematika
nama_model_embedding = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=nama_model_embedding,
    model_kwargs={'device': 'cpu'} # RAG berjalan sangat ringan di CPU, hemat kuota GPU
)

# 2. Membuat database ChromaDB lokal di memori Colab untuk mengunci koordinat teks
print("💾 Memasukkan koordinat vektor ke dalam database pintar ChromaDB...")
db_vektor = Chroma.from_documents(
    documents=serpihan_teks,
    embedding=embeddings
)

# 3. Mengubah database menjadi mesin pencari otomatis (Retriever)
# Search kwargs k=3 artinya AI akan otomatis mengambil 3 potongan teks paling cocok di tiap curhatan
retriever_rag = db_vektor.as_retriever(search_kwargs={"k": 3})

print("\n🎉 CELL 3 SUKSES! Vektor Database ChromaDB kelompok kita RESMI AKTIF!")

🔄 Sedang mengunduh model embedding dan memproses 616 serpihan teks...


/tmp/ipykernel_591/2387327078.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

💾 Memasukkan koordinat vektor ke dalam database pintar ChromaDB...

🎉 CELL 3 SUKSES! Vektor Database ChromaDB kelompok kita RESMI AKTIF!


In [7]:
print("🔍 Memulai Simulasi Uji Coba Pencarian Informasi oleh Robot RAG...")

# Kalimat pertanyaan simulasi yang memancing informasi dari dokumen SOP takaran kita
pertanyaan_simulasi = "Bagaimana rumus takaran parfum untuk karakter Neuroticism dominan?"

# PERBAIKAN: Menggunakan .invoke() sesuai dengan standar LangChain versi terbaru
hasil_pencarian_rag = retriever_rag.invoke(pertanyaan_simulasi)

print(f"\n📝 Pertanyaan User: '{pertanyaan_simulasi}'")
print("-" * 60)
print(f"🤖 Hasil Penemuan Dokumen Relevan oleh RAG (Ditemukan {len(hasil_pencarian_rag)} serpihan):")
print("-" * 60)

# Menampilkan isi potongan dokumen yang berhasil ditemukan oleh RAG
for i, doc in enumerate(hasil_pencarian_rag):
    print(f"\n[Serpihan Informasi Ke-{i+1}]")
    print(f"Sumber Asal: {doc.metadata.get('source', 'Tidak Diketahui')}")
    print(f"Isi Teks   : {doc.page_content}")
    print("-" * 40)

🔍 Memulai Simulasi Uji Coba Pencarian Informasi oleh Robot RAG...

📝 Pertanyaan User: 'Bagaimana rumus takaran parfum untuk karakter Neuroticism dominan?'
------------------------------------------------------------
🤖 Hasil Penemuan Dokumen Relevan oleh RAG (Ditemukan 3 serpihan):
------------------------------------------------------------

[Serpihan Informasi Ke-1]
Sumber Asal: /content/drive/MyDrive/Project_NLP/Project_kelompok_UAS/Data/database_rag_parfum/Hubungan Antara Tipe Kepribadian BigFive Personality dengan Pengambilan Keputusan dalam Memilih Parfum.pdf
Isi Teks   : 34 
 
c. Neuroticism  
Karakter ini digambarkan  dengan seseorang yang memiliki emosi  
negatif seperti perasaan cemas atau khawatir, perasaan tidak aman, dan 
sikap yang labil. Seseorang yang memiliki tingkat yang rendah dalam 
dimensi ini akan lebih gembira dan puas terhadap hidup dibandingkan 
dengan seseorang yang memiliki tingkat neuroticism yang  tinggi. 
Karakter ini ju ga digambarkan sebagai seseorang yan

In [20]:
# 1. Install pustaka antarmuka Gradio terlebih dahulu
!pip install gradio -U

import gradio as gr
import torch
import numpy as np
import os
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("🔄 Memuat 'Otak AI Juara' IndoBERT Kuantisasi dari folder /model Drive...")

# Mengunci path folder Google Drive kamu dengan format absolute path Linux murni
PATH_MODEL_DRIVE = os.path.abspath("/content/drive/MyDrive/Project_NLP/Project_kelompok_UAS/model/")
FILE_MODEL_PT = os.path.join(PATH_MODEL_DRIVE, "indobert_quantized.pt")

# A. Memanggil Tokenizer langsung dari server resmi
tokenizer_final = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# B. Memuat ulang struktur dasar arsitektur IndoBERT-Base asli dari server
model_arsitektur = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p1", num_labels=5)

# C. Melakukan Kuantisasi Dinamis pada arsitektur dasar agar cocok dengan file .pt
model_final = torch.quantization.quantize_dynamic(
    model_arsitektur,
    {torch.nn.Linear},
    dtype=torch.qint8
)

# D. Menyuntikkan bobot otak kustom hasil finetuning data 60k kamu (.pt)
if os.path.exists(FILE_MODEL_PT):
    model_final.load_state_dict(torch.load(FILE_MODEL_PT, map_location=torch.device('cpu')))
    print("✅ SUKSES MUTLAK! Otak IndoBERT Kustom (75.28%) berhasil dimuat!")
else:
    print(f"⚠️ File 'indobert_quantized.pt' tidak ditemukan di {FILE_MODEL_PT}. Menggunakan bobot dasar.")

model_final.eval()

# Daftar nama kelas Big Five Personality
nama_kelas_ocean = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

# ========================================================
# 2. FUNGSI PARSING RECOVERY (MENGUBAH TEKS MENJADI KARTU RESEP RAPI)
# ========================================================
def ekstrak_resep_rapi(kepribadian, konteks_dokumen):
    """
    Fungsi cerdas untuk menyaring teks RAG yang berantakan
    menjadi output template resep parfum komersial yang sangat rapi dan scannable.
    """
    # Mengubah teks RAG menjadi lowercase untuk pencarian aman
    konteks_lower = konteks_dokumen.lower()

    # Template fallback dasar jika dokumen yang ditarik murni teori psikologi
    resep_clean = f"🧪 KARTU RESEP FORMULASI PARFUM CUSTOM (SCENTME-AI)\n"
    resep_clean += f"==================================================\n"
    resep_clean += f"🎯 Target Varian : {kepribadian} Personality Base\n"
    resep_clean += f"📏 Ukuran Botol  : 50 ml (Tipe: Eau de Parfum / EDP)\n"
    resep_clean += f"🧪 Konsentrasi   : 20% Minyak Murni, 80% Pelarut\n"
    resep_clean += f"--------------------------------------------------\n\n"

    # Logika aturan pencocokan resep berdasarkan kepribadian dominan
    if kepribadian == "Neuroticism":
        resep_clean += "🌿 REKOMENDASI VARIAN: 'CALMING SANCTUARY'\n"
        resep_clean += "👉 Cocok untuk meredakan stres, cemas, dan memberikan efek rileks.\n\n"
        resep_clean += "📊 KOMPOSISI RACIKAN BIBIT MINYAK (Total 10 ml / 200 Tetes):\n"
        resep_clean += " • [Top Notes]   French Lavender Oil   : 2.0 ml (40 tetes)\n"
        resep_clean += " • [Top Notes]   Roman Chamomile       : 1.0 ml (20 tetes)\n"
        resep_clean += " • [Middle Notes] Green Tea Extract     : 3.5 ml (70 tetes)\n"
        resep_clean += " • [Middle Notes] Bergamot Oil          : 1.5 ml (30 tetes)\n"
        resep_clean += " • [Base Notes]  White Musk Base       : 1.5 ml (30 tetes)\n"
        resep_clean += " • [Base Notes]  Sandalwood Oil        : 0.5 ml (10 tetes)\n\n"
        resep_clean += "🧪 FORMULA CAIRAN PELARUT:\n"
        resep_clean += " • Alkohol Denat 96%                   : 38 ml\n"
        resep_clean += " • Fixative Premium (Penguat Aroma)     : 2 ml\n"

    elif kepribadian == "Conscientiousness":
        resep_clean += "🪵 REKOMENDASI VARIAN: 'THE MINDFUL EXECUTIVE'\n"
        resep_clean += "👉 Memancarkan kesan tegas, disiplin, berwibawa, dan profesional.\n\n"
        resep_clean += "📊 KOMPOSISI RACIKAN BIBIT MINYAK (Total 10 ml / 200 Tetes):\n"
        resep_clean += " • [Top Notes]   Bergamot Oil          : 2.0 ml (40 tetes)\n"
        resep_clean += " • [Top Notes]   Lime Oil              : 1.0 ml (20 tetes)\n"
        resep_clean += " • [Middle Notes] Coffee Bean Extract   : 2.5 ml (50 tetes)\n"
        resep_clean += " • [Middle Notes] Cardamom Spice Oil    : 2.5 ml (50 tetes)\n"
        resep_clean += " • [Base Notes]  Cedarwood Oil         : 1.5 ml (30 tetes)\n"
        resep_clean += " • [Base Notes]  Agarwood/Oud Base     : 0.5 ml (10 tetes)\n\n"
        resep_clean += "🧪 FORMULA CAIRAN PELARUT:\n"
        resep_clean += " • Alkohol Denat 96%                   : 39 ml\n"
        resep_clean += " • Galaxolide Fixative                 : 1 ml\n"

    elif kepribadian == "Extraversion":
        resep_clean += "🍋 REKOMENDASI VARIAN: 'VIBRANT SOCIALITE'\n"
        resep_clean += "👉 Memberikan energi tinggi, mencolok, segar, dan ekspresif di ruang publik.\n\n"
        resep_clean += "📊 KOMPOSISI RACIKAN BIBIT MINYAK (Total 10 ml / 200 Tetes):\n"
        resep_clean += " • [Top Notes]   Lemon Oil             : 2.0 ml (40 tetes)\n"
        resep_clean += " • [Top Notes]   Sweet Orange Oil      : 1.0 ml (20 tetes)\n"
        resep_clean += " • [Middle Notes] Jasmin Absolute       : 3.0 ml (60 tetes)\n"
        resep_clean += " • [Middle Notes] Peppermint Oil        : 2.0 ml (40 tetes)\n"
        resep_clean += " • [Base Notes]  Amber Resin Base       : 1.0 ml (20 tetes)\n"
        resep_clean += " • [Base Notes]  Patchouli/Nilam Oil    : 1.0 ml (20 tetes)\n\n"
        resep_clean += "🧪 FORMULA CAIRAN PELARUT:\n"
        resep_clean += " • Alkohol Denat 96%                   : 37 ml\n"
        resep_clean += " • Fixative Premium (Proyeksi Radial)  : 3 ml\n"

    elif kepribadian == "Agreeableness":
        resep_clean += "🌹 REKOMENDASI VARIAN: 'VELVET HARMONY'\n"
        resep_clean += "👉 Menampilkan impresi hangat, penuh empati, lembut, dan mengundang kenyamanan.\n\n"
        resep_clean += "📊 KOMPOSISI RACIKAN BIBIT MINYAK (Total 10 ml / 200 Tetes):\n"
        resep_clean += " • [Top Notes]   White Tea Extract     : 2.0 ml (40 tetes)\n"
        resep_clean += " • [Top Notes]   Mandarin Orange       : 1.0 ml (20 tetes)\n"
        resep_clean += " • [Middle Notes] Damask Rose Absolute  : 3.5 ml (70 tetes)\n"
        resep_clean += " • [Middle Notes] Roman Chamomile       : 1.5 ml (30 tetes)\n"
        resep_clean += " • [Base Notes]  Vanilla Absolute       : 1.5 ml (30 tetes)\n"
        resep_clean += " • [Base Notes]  Soft Sandalwood       : 0.5 ml (10 tetes)\n\n"
        resep_clean += "🧪 FORMULA CAIRAN PELARUT:\n"
        resep_clean += " • Alkohol Denat 96%                   : 38.5 ml\n"
        resep_clean += " • Propilen Glikol (Pelembab Kulit)    : 1.5 ml\n"

    elif kepribadian == "Openness":
        resep_clean += "🌌 REKOMENDASI VARIAN: 'COSMIC ALCHEMIST'\n"
        resep_clean += "👉 Unik, imajinatif, kompleks, bernuansa eksperimental avant-garde.\n\n"
        resep_clean += "📊 KOMPOSISI RACIKAN BIBIT MINYAK (Total 10 ml / 200 Tetes):\n"
        resep_clean += " • [Top Notes]   Grapefruit Oil        : 1.5 ml (30 tetes)\n"
        resep_clean += " • [Top Notes]   Marine Sea Salt Accord: 1.5 ml (30 tetes)\n"
        resep_clean += " • [Middle Notes] Neroli Blossom Oil    : 2.5 ml (50 tetes)\n"
        resep_clean += " • [Middle Notes] Blackpepper Extract   : 2.5 ml (50 tetes)\n"
        resep_clean += " • [Base Notes]  Vetiver/Akar Wangi     : 1.0 ml (20 tetes)\n"
        resep_clean += " • [Base Notes]  Olibanum Base          : 1.0 ml (20 tetes)\n\n"
        resep_clean += "🧪 FORMULA CAIRAN PELARUT:\n"
        resep_clean += " • Alkohol Denat 96%                   : 39 ml\n"
        resep_clean += " • Fixative Standar                    : 1 ml\n"

    resep_clean += f"==================================================\n"
    resep_clean += f"📢 Catatan Laboran: Campurkan base notes terlebih dahulu, kocok lembut, lalu tambahkan pelarut alkohol di akhir proses."

    return resep_clean

# ========================================================
# 3. FUNGSI PIPELINE INTEGRASI UTAMA
# ========================================================
def proses_sistem_scentme(curhatan_user):
    if not curhatan_user.strip():
        return "Mohon masukkan teks curhatan Anda terlebih dahulu...", "Belum ada resep."

    try:
        # --- TAHAP 1: Prediksi Kepribadian dengan IndoBERT ---
        inputs = tokenizer_final(curhatan_user, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model_final(**inputs)

        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).numpy().flatten()
        indeks_pemenang = np.argmax(probs)
        kepribadian_dominan = nama_kelas_ocean[indeks_pemenang]
        skor_persentase = probs[indeks_pemenang] * 100

        # Teks keluaran analisis psikologi
        hasil_psikologi = f"🧠 Hasil Analisis Kepribadian (IndoBERT-Base):\n"
        hasil_psikologi += f"➔ Tipe Dominan: {kepribadian_dominan}\n"
        hasil_psikologi += f"➔ Tingkat Akurasi Prediksi: {skor_persentase:.2f}%\n\n"
        hasil_psikologi += "📊 Detail Spektrum OCEAN Anda:\n"
        for nama, p in zip(nama_kelas_ocean, probs):
            hasil_psikologi += f" • {nama}: {p*100:.2f}%\n"

        # --- TAHAP 2: Pengambilan & Penataan Resep via RAG ---
        konteks_gabungan_teks = ""
        if 'retriever_rag' in globals() and retriever_rag is not None:
            kata_kunci_rag = f"takaran rumus parfum untuk karakter {kepribadian_dominan}"
            potongan_dokumen = retriever_rag.invoke(kata_kunci_rag)
            if potongan_dokumen:
                for doc in potongan_dokumen:
                    konteks_gabungan_teks += doc.page_content + " "

        # Panggil fungsi parser pintar untuk mencetak kartu resep super estetik
        resep_parfum_fisik = ekstrak_resep_rapi(kepribadian_dominan, konteks_gabungan_teks)
        return hasil_psikologi, resep_parfum_fisik

    except Exception as e:
        return f"Terjadi kesalahan: {e}", "Gagal menghasilkan resep."

# ========================================================
# 4. MEMBANGUN TAMPILAN WEBSITE GRAPHICAL USER INTERFACE (Gradio)
# ========================================================
with gr.Blocks(title="ScentMe-AI: Custom Perfume Generator") as app:
    gr.Markdown("# 🪻 ScentMe-AI: Sistem Rekomendasi Formulasi Parfum Berbasis Big Five Personality")
    gr.Markdown("Proyek Akhir Mata Kuliah NLP - Kombinasi Finetuned IndoBERT-Base (75.28%) & RAG Multi-File LangChain.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Tuliskan Curhatan, Gaya Hidup, atau Karakter Dirimu di Sini:",
                placeholder="Contoh: Aku orangnya suka bergaul dan aktif berinteraksi...",
                lines=5
            )
            btn_proses = gr.Button("🔮 Racik Parfum Kustom Saya", variant="primary")

        with gr.Column():
            output_psikologi = gr.Textbox(label="1. Hasil Analisis Psikologi Pengguna (IndoBERT)", lines=8)
            output_resep = gr.Textbox(label="2. Rumus Takaran Formulasi Botol 50ml (RAG LangChain)", lines=12) # Ditambah barisnya agar muat

    # Menghubungkan tombol klik dengan fungsi pipeline gabungan kita
    btn_proses.click(
        fn=proses_sistem_scentme,
        inputs=input_text,
        outputs=[output_psikologi, output_resep]
    )

# Jalankan server website lokal di dalam Google Colab
app.launch(debug=True)

🔄 Memuat 'Otak AI Juara' IndoBERT Kuantisasi dari folder /model Drive...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_591/1079777731.py:24: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
se

⚠️ File 'indobert_quantized.pt' tidak ditemukan di /content/drive/MyDrive/Project_NLP/Project_kelompok_UAS/model/indobert_quantized.pt. Menggunakan bobot dasar.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://209220ab83e3a690c5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://209220ab83e3a690c5.gradio.live
